# Stage 3 (recognition + Markush branch) on Google Colab

Runtime: **Runtime > Change runtime type > T4 GPU** before running anything.

Steps: 1 install conda (the runtime restarts, which is expected) · 2 get the group repo · 3 build the stage 3 environment · 4 run the tests · 5 run the sample with the real model · 6 optional: run the whole chain on the six course papers · 7 download the results.


## 1. Install conda (run once; the kernel restarts afterwards, do not rerun)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Get the group repo
Either set `REPO_URL` (public repo), or leave it empty and upload the repo ZIP when asked (GitHub: Code > Download ZIP).

In [ ]:
import os, glob, subprocess
REPO_URL = "https://github.com/Showrish/BracketR-XN-Pipeline.git"   # leave "" to upload a ZIP instead

os.chdir("/content")
if REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "repo_unzipped"], check=True)
    inner = [d for d in glob.glob("repo_unzipped/*") if os.path.isdir(d)][0]
    os.rename(inner, "repo")
os.chdir("/content/repo")
assert os.path.isdir("stage3_recognition") and os.path.isdir("stage1_2_detection"), "repo layout not found"
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null
!which pdftotext pdftoppm
!ls

## 3. Build the stage 3 environment (MolScribe + RDKit + tests)

In [ ]:
!conda env create -q -f stage3_recognition/envs/stage3_molscribe.yml
!conda env list

## 4. Run the tests

In [ ]:
!conda run --no-capture-output -n xn-stage3-molscribe python -m pytest stage3_recognition/tests -q

## 5. Sample run with the real model
Writes `stage3_recognition/examples/sample_output/` and prints exact matches against the known answers.

In [ ]:
!conda run --no-capture-output -n xn-stage3-molscribe python -m stage3_recognition.examples.run_sample
!cat stage3_recognition/examples/sample_output/stage12_sample/predictions_molscribe.csv
!cat stage3_recognition/examples/sample_output/markush_sample/markush_scaffolds.csv
!cat stage3_recognition/examples/sample_output/markush_sample/markush_compounds.csv

## 6. Optional: the whole chain on the six course papers
Upload `Dataset_2.zip` (the course PDFs and answer key) when asked. Stage 1-2 runs in Colab's own Python, stage 3 in its environment.

In [ ]:
from google.colab import files
uploaded = files.upload()
!unzip -q -o "{next(iter(uploaded))}" -x "__MACOSX/*" -d /content/course
!pip -q install opencv-python-headless scikit-image pdf2image pyyaml
!python -m stage1_2_detection.cli --pdf-dir /content/course/Dataset --out /content/out/stage12
!conda run --no-capture-output -n xn-stage3-molscribe python -m stage3_recognition.cli --manifest /content/out/stage12/manifest.csv --out /content/out/stage3 --pdf-dir /content/course/Dataset
!conda run --no-capture-output -n xn-stage3-molscribe python -m stage3_recognition.examples.check_course_papers --stage3 /content/out/stage3 --answer-key "/content/course/Dataset/ground_truth_SMILES_confirmed_1,2,4,6,16,19.csv"
!conda run --no-capture-output -n xn-stage3-molscribe python -m stage3_recognition.examples.check_tables --pdf-dir /content/course/Dataset --out /content/out/stage3/table_check.json

## 7. Download the results
The prepared crops of the course papers are left out on purpose: they are figures from the papers and should not go into a public repo.

In [ ]:
!mkdir -p /content/out
!cd /content && zip -qr stage3_results.zip repo/stage3_recognition/examples/sample_output out/stage3 -x "out/stage3/prepared/*"
from google.colab import files
files.download("/content/stage3_results.zip")